Agentic RAG with LangGraph

In [1]:
import os
from typing import TypedDict,List
from langgraph.graph import StateGraph,END
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\DELL\Desktop\TraditionalRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\DELL\AppData\Local\Temp\ipykernel_8320\1808813910.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# Set Your OpenAI API key
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

# Initialize models
llm=ChatOpenAI(model="gpt-4o-mini",temperature=0)
embeddings=OpenAIEmbeddings()


In [4]:
llm

ChatOpenAI(output_version=None, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000029B8CB95A90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000029B8CB96510>, root_client=<openai.OpenAI object at 0x0000029B8C2BC440>, root_async_client=<openai.AsyncOpenAI object at 0x0000029B8CB96270>, model_name='gpt-4o-mini', temperature=0.0, 

State Definition

In [5]:
class AgentState(TypedDict):
    question: str 
    documents: List[Document]
    answer: str 
    needs_retrieval: bool 

In [6]:
### Sample Documents and Vector Store
# Sample Documents for demonstartion

sample_texts = [
    "LangGraph is a framework for building stateful, multi-agent AI applications using graphs.",
    "It extends LangChain by allowing developers to define workflows as nodes and edges in a graph.",
    "LangGraph supports cyclical workflows, enabling agents to revisit previous steps when needed.",
    "It provides built-in state management, allowing information to persist across different stages of execution.",
    "LangGraph is commonly used for creating AI agents, multi-step reasoning systems, and complex orchestration pipelines.",
    "It integrates seamlessly with LangChain components such as LLMs, tools, retrievers, and memory."
]

In [9]:
documents=[Document(page_content=text) for text in sample_texts]
documents

### Create Vector Store
vectorstore=FAISS.from_documents(documents,embeddings)
retriever=vectorstore.as_retriever(k=3)

Agents Function

In [10]:
def decide_retrieval(state: AgentState) -> AgentState:
    """
    Decide if we need to retrieve documents based on the question
    """
    question=state["question"]

    # Simple heuristic: if question contains keywords, retrieve
    retrieval_keywords=["what","how","explain","describe","tell me"]
    needs_retrieval=any(keyword in question.lower() for keyword in retrieval_keywords)

    return {**state, "needs_retrieval": needs_retrieval}

In [11]:
def retrieve_documents(state: AgentState) -> AgentState:
    """
    Retrieve relevant documents based on the question 
    """
    question=state['question']
    documents=retriever.invoke(question)

    return {**state, "documents": documents}

In [21]:
def generate_answer(state: AgentState) -> AgentState:
    """
    Generate an answer using the retrieved documents or direct response
    """
    question = state["question"]
    documents = state.get("documents", [])
    if documents:
        context = "\n\n".join([doc.page_content for doc in documents])
        prompt = f"""
        Based on the following context, answer the question.
        Context:{context}
        Question:{question}
        Answer:
        """
    else:
        prompt = f"Answer the following question: {question}"
    # Call LLM for BOTH cases
    response = llm.invoke(prompt)
    # Extract text
    answer = response.content
    return {**state,"answer": answer}

Conditional Logic

In [22]:
def should_retrieve(state: AgentState) -> AgentState:
    """
    Determine the next step based on retrieval decision
    """
    if state["needs_retrieval"]:
        return "retrieve"
    else:
        return "generate"

build the graph

In [23]:
# Create the State Graph
workflow=StateGraph(AgentState)

# Add nodes
workflow.add_node("decide",decide_retrieval)
workflow.add_node("retrieve",retrieve_documents)
workflow.add_node("generate",generate_answer)

# Set Entry Point
workflow.set_entry_point("decide")

# Add Conditional edges
workflow.add_conditional_edges(
    "decide",
    should_retrieve,
    {
        "retrieve": "retrieve",
        "generate": "generate"
    }
)

# Add the edges
workflow.add_edge("retrieve","generate")
workflow.add_edge("generate",END)

In [24]:
app = workflow.compile()

test the Agentic System

In [25]:
def ask_question(question: str):
    """
    Helper Function to ask a question and get an answer 
    """
    initial_state={
        "question": question,
        "documents": [],
        "answer": "",
        "needs_retrieval": False
    }

    result=app.invoke(initial_state)
    return result

In [26]:
# Test with a question that should trigger retrieval
question1= "what is Langgraph"
result1=ask_question(question1)

In [27]:
result1

{'question': 'what is Langgraph',
 'documents': [Document(id='a43e2bbf-42ef-47fe-90b6-b3ebe516df38', metadata={}, page_content='LangGraph is a framework for building stateful, multi-agent AI applications using graphs.'),
  Document(id='5a868caa-a7aa-4347-980a-135e60643865', metadata={}, page_content='LangGraph is commonly used for creating AI agents, multi-step reasoning systems, and complex orchestration pipelines.'),
  Document(id='79aca12d-50a1-4be3-ac15-8593e5eff349', metadata={}, page_content='It extends LangChain by allowing developers to define workflows as nodes and edges in a graph.'),
  Document(id='7ae5c039-d1bd-43d8-bea6-c54abd201a6c', metadata={}, page_content='It integrates seamlessly with LangChain components such as LLMs, tools, retrievers, and memory.')],
 'answer': 'LangGraph is a framework designed for building stateful, multi-agent AI applications using graphs. It enables the creation of AI agents, multi-step reasoning systems, and complex orchestration pipelines by